In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from IPython.display import display

# Ensure the notebook can find the /src directory
root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

In [ ]:
from src.data_loader import VesselDataLoader
from src.visualizer import (
    plot_block_space, 
    plot_mode_statistics, 
    plot_series,
    plot_scenario_tradeoff_space,
    plot_trajectory_folium
)
from src.data_processing import engineer_telemetry_features
from src.mission_profiler import MissionProfiler, ScenarioManager

In [ ]:
filenames = ["Wembley_voy_236.csv" ,"Rotherhithe_voy_179.csv", "your_5_month_data.csv"]
processed_data_cache = {}

# Load data into memory once
for filename in filenames:
    print(f"Ingesting {filename} from PROCESSED folder...")
    processed_data_file = root_path / "data" / "processed" / filename
    
    # Skip if we haven't labeled it yet
    if not processed_data_file.exists():
        print(f" -> Skipped. {filename} not found in processed directory.")
        continue
        
    loader = VesselDataLoader(processed_data_file)
    processed_data_cache[filename] = loader.load_and_clean()

In [ ]:
filtered_registry_entries = []
profiler = MissionProfiler(speed_threshold=1.0)

for filename, proc_df in processed_data_cache.items():
    print(f"\nProcessing default features for {filename}...")
    
    # Applying low-pass filtering automatically computes the inline battery specs
    filtered_df = engineer_telemetry_features(proc_df, filter_method='butter')
    
    # Inspect battery limits for the hardware envelope tracking
    battery_metrics = filtered_df.attrs.get('battery_specs', {})
    print(f"  -> Worst-Case Causal Power Peak Required: {battery_metrics.get('worst_case_power_peak_kW', 0.0):.1f} kW")
    print(f"  -> Min Energy Capacity Excursion Integral: {battery_metrics.get('min_capacity_excursion_kWh', 0.0):.1f} kWh")
    
    # Generate Low-Pass Filtered Block Registries (Target Architecture Base)
    filtered_registry_df = profiler.generate_block_registry(
        filtered_df, 
        source_file_name=filename, 
        merge_loitering=False, 
        unify_port_ops=False
    )
    filtered_registry_entries.append(filtered_registry_df)
    
    # Standard validation diagnostic visualizations
    fig, axes = plot_series(filtered_df, ['AE_POWER(kW)'])
    plt.show() 
    display(plot_trajectory_folium(filtered_df))

# Concatenate all blocks and generate mathematically consistent global stats
if filtered_registry_entries:
    combined_filtered_registry = pd.concat(filtered_registry_entries, ignore_index=True)
    combined_filtered_stats = profiler.extract_global_statistics(combined_filtered_registry)
else:
    print("No datasets were successfully processed.")

In [ ]:
cutoff_presets = [0.40, 0.20, 0.10, 0.05, 0.02, 0.01]

# NOTE: Updated to standard lowercase mode names
evaluation_runs = {
    "Full Spectrum": [],
    "Transit & Maneuvering Only": ['port_loading', 'port_unloading']
}

for filename, proc_df in processed_data_cache.items():
    print(f"\n{'='*30}\nBATTERY PARAMETER SWEEP: {filename}\n{'='*30}")
    
    # Build the structural baseline strictly from memory
    raw_features_df = engineer_telemetry_features(proc_df.copy(), filter_method='raw')
    
    # Note: Since STATUS is now baked into the file, we just map it.
    # We no longer need classify_modes() to guess the statuses!
    base_classified = raw_features_df.copy()
    if 'MODE' not in base_classified.columns:
        base_classified['MODE'] = base_classified['STATUS']

    for run_name, ignore_list in evaluation_runs.items():
        print(f"\n--- CONFIGURATION: {run_name} ---")
        battery_sweep_records = []
        
        for fc in cutoff_presets:
            sweep_filtered_df = engineer_telemetry_features(
                base_classified, 
                filter_method='butter', 
                cutoff=fc, 
                ignore_modes=ignore_list
            )
            
            specs = sweep_filtered_df.attrs.get('battery_specs', {})
            if specs:
                installed_capacity_buffer_60pc = specs['min_capacity_excursion_kWh'] / 0.60
                battery_sweep_records.append({
                    'Cutoff (fc)': fc,
                    'Max Inverter (kW)': round(specs['worst_case_power_peak_kW'], 1),
                    'Min Usable (kWh)': round(specs['min_capacity_excursion_kWh'], 1),
                    'Est Pack (kWh)': round(installed_capacity_buffer_60pc, 1)
                })
        
        print(pd.DataFrame(battery_sweep_records).to_string(index=False))

In [ ]:
if filtered_registry_entries:
    # 1. Plot Block Workspace
    fig_bricks_fatigue = plot_block_space(combined_filtered_registry, y_axis_metric='Relative_Fatigue_Activity_Rate')
    fig_bricks_fatigue.show()

    # 2. Plot Aggregated Mode Statistics
    fig_stats_fatigue = plot_mode_statistics(combined_filtered_stats, y_axis_metric='Relative_Fatigue_Activity_Rate', print_table=False)
    fig_stats_fatigue.show()

    # 3. Instantiate Scenario Manager & Run Pipeline
    manager = ScenarioManager(registry_df=combined_filtered_registry, global_stats=combined_filtered_stats)
    compiled_scenario_outputs = manager.run_full_scenario_pipeline(total_target_h=1000.0, print_summary=True)

    # 4. Render Final Scenario Optimization Tradeoff Chart
    fig_tradeoff = plot_scenario_tradeoff_space(compiled_scenario_outputs)
    fig_tradeoff.show()